In [ ]:
import gurobipy as gp
from gurobipy import GRB
from data_loader import load_data
from collections import defaultdict
import time
import os
import pandas as pd
experiment_start_time = time.perf_counter()


In [ ]:
EXPERIMENT_NO = 8

DEFAULT_HARD_TIME_LIMIT = 600
DEFAULT_SOFT_TIME_LIMIT = 2700

EXPERIMENT_PROFILES = {
    0: {
        "name": "0_only_strong",
        "unfavorable": 0,
        "teacher_gap": 0,
        "class_gap": 0,
        "subject_spread": 0,
        "teacher_min_day": 0,
        "soft_time_limit": 60,
    },
    1: {
        "name": "1_balanced",
        "unfavorable": 1,
        "teacher_gap": 1,
        "class_gap": 1,
        "subject_spread": 1,
        "teacher_min_day": 1,
    },
    2: {
        "name": "2_unfavorable",
        "unfavorable": 10,
        "teacher_gap": 1,
        "class_gap": 1,
        "subject_spread": 1,
        "teacher_min_day": 1,
    },
    3: {
        "name": "3_teacher_gap",
        "unfavorable": 1,
        "teacher_gap": 10,
        "class_gap": 1,
        "subject_spread": 1,
        "teacher_min_day": 1,
    },
    4: {
        "name": "4_class_gap",
        "unfavorable": 1,
        "teacher_gap": 1,
        "class_gap": 10,
        "subject_spread": 1,
        "teacher_min_day": 1,
    },
    5: {
        "name": "5_subject_spread",
        "unfavorable": 1,
        "teacher_gap": 1,
        "class_gap": 1,
        "subject_spread": 10,
        "teacher_min_day": 1,
    },
    6: {
        "name": "6_teacher_min_day",
        "unfavorable": 1,
        "teacher_gap": 1,
        "class_gap": 1,
        "subject_spread": 1,
        "teacher_min_day": 10,
    },
    7: {
        "name": "7_best_long",
        "unfavorable": 1,
        "teacher_gap": 5,
        "class_gap": 5,
        "subject_spread": 4,              
        "teacher_min_day": 2,
        "soft_time_limit": 28800,
    },
    8: {
        "name": "8_best_long2",
        "unfavorable": 1,
        "teacher_gap": 8,
        "class_gap": 8,
        "subject_spread": 2,              
        "teacher_min_day": 2,
        "soft_time_limit": 43200,
    },
}

CURRENT_EXPERIMENT = EXPERIMENT_PROFILES[EXPERIMENT_NO]
EXPERIMENT_NAME = CURRENT_EXPERIMENT["name"]
HARD_TIME_LIMIT = CURRENT_EXPERIMENT.get("hard_time_limit", DEFAULT_HARD_TIME_LIMIT)
SOFT_TIME_LIMIT = CURRENT_EXPERIMENT.get("soft_time_limit", DEFAULT_SOFT_TIME_LIMIT)

print(f"Eksperiments: {EXPERIMENT_NO} ({EXPERIMENT_NAME})")
print(f"Laika limits: {HARD_TIME_LIMIT}s + {SOFT_TIME_LIMIT}s")


In [ ]:
data = load_data()

T = data["T"]
C = data["C"]
R = data["R"]
S = data["S"]
P = data["P"]
D = data["D"]
L = data["L"]

In [ ]:
Avail = data["Avail"]
Req = data["Req"]
Qual = data["Qual"]
Fit = data["Fit"]
Suit = data["Suit"]
Unf = data["Unf"]
Participants = data["Participants"]
ShareMode = data["ShareMode"]
GroupIndex = data["GroupIndex"]
Instance = data["Instance"]
StreamId = data["StreamId"]
SyncKey = data["SyncKey"]
PartitionKey = data["PartitionKey"]
VariantId = data["VariantId"]
PartitionVariants = data["PartitionVariants"]
MaxDay = data["MaxDay"]
MinDay = data["MinDay"]
MaxPerDay = data["MaxPerDay"]
MinStart = data["MinStart"]
MaxGroup = data["MaxGroup"]

In [ ]:
def Class(l): return l[1]
def Subj(l):  return l[2]
def SharedKey(l): return l[3]
def AssignedTeacher(l): return l[4]
def GroupCount(l): return l[5]

In [ ]:
model = gp.Model("timetable")

model.setParam("MIPFocus", 1)
model.setParam("MIPGap", 0.12)
model.setParam("TimeLimit", HARD_TIME_LIMIT)
model.setParam("Threads", 6)
model.setParam("NodefileStart", 6.0)
model.setParam("Presolve", 2)
model.setParam("Heuristics", 0.5)
model.setParam("OutputFlag", 0)


In [ ]:
teacher_required_hours = {}
for l in L:
    t = l[4]
    teacher_required_hours[t] = teacher_required_hours.get(t, 0) + 1

for t, req in teacher_required_hours.items():
    min_feasible_max_day = min((req + len(D) - 1) // len(D), len(P))
    if MaxDay[t] < min_feasible_max_day:
        MaxDay[t] = min_feasible_max_day

    if req > 35: 
        for d in D:
            for p in P:
                Avail[(t, d, p)] = 1

In [ ]:
unique_lesson_ids = sorted(set(l[0] for l in L))
lesson_classes = {l[0]: l[1] for l in L}
lesson_subjects = {l[0]: l[2] for l in L}
lesson_teachers = {l[0]: l[4] for l in L}

valid_combos = [
    (l_id, r, d, p)
    for l_id in unique_lesson_ids
    for r in R
    for d in D
    for p in P
    if Fit.get((r, l_id), 1) == 1
    and Suit.get((r, l_id), 1) == 1
    and Avail.get((lesson_teachers[l_id], d, p), 0) == 1
]

x = model.addVars(valid_combos, vtype=GRB.BINARY, name="x")
variant_choices = [
    (partition_key, variant_id)
    for partition_key, variants in PartitionVariants.items()
    for variant_id in variants
]
y = model.addVars(variant_choices, vtype=GRB.BINARY, name="variant")

model.update()

## Stingrie ierobežojumi


### H0. Sadalījuma varianta izvēle

In [ ]:
for partition_key, variants in PartitionVariants.items():
    model.addConstr(
        gp.quicksum(y[partition_key, variant_id] for variant_id in variants) == 1,
        name=f"H0_variant_choice_{partition_key}"
    )

### H1. Katra nodarbība notiek tieši vienreiz

In [ ]:
for l_id in unique_lesson_ids:
    variant_id = VariantId.get(l_id, "")
    if variant_id:
        model.addConstr(
            gp.quicksum(x.select(l_id, '*', '*', '*')) == y[PartitionKey[l_id], variant_id],
            name=f"H1_variant_lesson_{l_id}"
        )
    else:
        model.addConstr(
            gp.quicksum(x.select(l_id, '*', '*', '*')) == 1,
            name=f"H1_lesson_once_{l_id}"
        )

### H2. Skolotāja vienlaicīguma aizliegums

In [ ]:
teacher_lesson_ids = defaultdict(list)
for l in L:
    teacher_lesson_ids[AssignedTeacher(l)].append(l[0])

for t, lesson_ids in teacher_lesson_ids.items():
    lesson_ids = sorted(set(lesson_ids))
    for d in D:
        for p in P:
            teacher_time_vars = []
            for l_id in lesson_ids:
                teacher_time_vars.extend(x.select(l_id, '*', d, p))
            if teacher_time_vars:
                model.addConstr(
                    gp.quicksum(teacher_time_vars) <= 1,
                    name=f"H2_teacher_time_{t}_{d}_{p}"
                )


### H3. Klases vienlaicīguma aizliegums

In [ ]:
class_lesson_ids_by_class = {}
for c in C:
    class_lesson_ids_by_class[c] = sorted(
        l_id for l_id in unique_lesson_ids
        if c in Participants.get(l_id, (lesson_classes[l_id],))
    )
class_subject_keys = sorted({
    (c, lesson_subjects[l_id])
    for c, lesson_ids in class_lesson_ids_by_class.items()
    for l_id in lesson_ids
})
class_subject_busy = model.addVars(
    [(c, s, d, p) for c, s in class_subject_keys for d in D for p in P],
    vtype=GRB.BINARY,
    name="class_subject_busy"
)

for c, s in class_subject_keys:
    subject_lesson_ids = [
        l_id for l_id in class_lesson_ids_by_class[c]
        if lesson_subjects[l_id] == s
    ]
    max_group = MaxGroup.get((c, s), 1)
    for d in D:
        for p in P:
            class_subject_vars = []
            for l_id in subject_lesson_ids:
                class_subject_vars.extend(x.select(l_id, '*', d, p))
            if class_subject_vars:
                model.addConstr(
                    gp.quicksum(class_subject_vars) <= max_group * class_subject_busy[c, s, d, p],
                    name=f"H3_subject_group_cap_{c}_{s}_{d}_{p}"
                )
                model.addConstr(
                    class_subject_busy[c, s, d, p] <= gp.quicksum(class_subject_vars),
                    name=f"H3_subject_busy_link_{c}_{s}_{d}_{p}"
                )

subjects_by_class = defaultdict(list)
for c, s in class_subject_keys:
    subjects_by_class[c].append(s)

for c, subjects in subjects_by_class.items():
    for d in D:
        for p in P:
            model.addConstr(
                gp.quicksum(class_subject_busy[c, s, d, p] for s in subjects) <= 1,
                name=f"H3_one_subject_per_class_time_{c}_{d}_{p}"
            )


### H4. Kabineta vienlaicīguma aizliegums

In [ ]:
for r in R:
    for d in D:
        for p in P:
            room_time_vars = x.select('*', r, d, p)
            if room_time_vars:
                model.addConstr(
                    gp.quicksum(room_time_vars) <= 1,
                    name=f"H4_room_time_{r}_{d}_{p}"
                )

### H5. Nedēļas stundu prasību izpilde

In [ ]:
required_pairs = sorted(Req.keys())
# H5 nodrošina nodarbību ģenerēšana un H1 ierobežojumi

### H6-H8. Pieejamība, kvalifikācija un kabineta piemērotība

Skolotāja pieejamība, kabineta piemērotība un ēkas/klases atbilstība tiek iekļauta mainīgo veidošanas brīdī ar `valid_combos`. Neatļautie `(lesson, room, day, period)` mainīgie netiek izveidoti.


In [ ]:
# H6, H7 un H8 ir iekļauti mainīgo ģenerēšanā

### H9. Sadalīto grupu sinhronizācija


In [ ]:
sync_groups = defaultdict(list)
for l in L:
    key = SyncKey.get(l[0], "")
    if key:
        sync_groups[key].append(l[0])

for key, lesson_ids in sync_groups.items():
    lesson_ids = sorted(set(lesson_ids))
    if len(lesson_ids) <= 1:
        continue
    ref_lid = lesson_ids[0]
    for other_lid in lesson_ids[1:]:
        for d in D:
            for p in P:
                model.addConstr(
                    gp.quicksum(x.select(ref_lid, '*', d, p)) == gp.quicksum(x.select(other_lid, '*', d, p)),
                    name=f"H9_sync_{key}_{ref_lid}_{other_lid}_{d}_{p}"
                )

### H10. Skolotāja maksimālais stundu skaits dienā

In [ ]:
for t, lesson_ids in teacher_lesson_ids.items():
    lesson_ids = sorted(set(lesson_ids))
    for d in D:
        teacher_day_vars = []
        for l_id in lesson_ids:
            for p in P:
                teacher_day_vars.extend(x.select(l_id, '*', d, p))
        if teacher_day_vars:
            model.addConstr(
                gp.quicksum(teacher_day_vars) <= MaxDay[t],
                name=f"H10_teacher_max_day_{t}_{d}"
            )

### H11. Klases maksimālais stundu skaits dienā


In [ ]:
class_busy = model.addVars(C, D, P, vtype=GRB.BINARY, name="class_busy")

for c, lesson_ids in class_lesson_ids_by_class.items():
    if not lesson_ids:
        continue
    for d in D:
        for p in P:
            class_time_vars = []
            for l_id in lesson_ids:
                class_time_vars.extend(x.select(l_id, '*', d, p))
            if class_time_vars:
                max_parallel = max(
                    MaxGroup.get((c, lesson_subjects[l_id]), 1)
                    for l_id in lesson_ids
                )
                model.addConstr(
                    class_busy[c, d, p] >= gp.quicksum(class_time_vars) / max_parallel,
                    name=f"H11_class_busy_link_{c}_{d}_{p}"
                )

for c in C:
    for d in D:
        model.addConstr(
            gp.quicksum(class_busy[c, d, p] for p in P) <= MaxPerDay[c],
            name=f"H11_class_max_day_{c}_{d}"
        )


### H12. Klases agrākais sākuma periods

In [ ]:
for c, lesson_ids in class_lesson_ids_by_class.items():
    if not lesson_ids:
        continue
    for d in D:
        for p in P:
            if p < MinStart[c]:
                class_time_vars = []
                for l_id in lesson_ids:
                    class_time_vars.extend(x.select(l_id, '*', d, p))
                if class_time_vars:
                    model.addConstr(
                        gp.quicksum(class_time_vars) == 0,
                        name=f"H12_class_min_start_{c}_{d}_{p}"
                    )


### Stingro ierobežojumu kopsavilkums


In [ ]:
model.update()
print(f"Stingrie ierobežojumi: {model.numConstrs}")

### Sākuma risinājums

Vispirms tiek atrasts viens derīgs risinājums bez mīkstajiem ierobežojumiem. Pēc tam tas tiek izmantots kā sākuma punkts mīksto ierobežojumu optimizācijai.


In [ ]:
model.setObjective(0, GRB.MINIMIZE)
model.setParam("SolutionLimit", 1)
model.setParam("TimeLimit", HARD_TIME_LIMIT)

model.optimize()

if model.SolCount == 0:
    raise RuntimeError("Stingrais modelis neatrada derīgu risinājumu")

for var in model.getVars():
    if var.X > 0.5:
        var.Start = 1
    else:
        var.Start = 0

model.setParam("SolutionLimit", 2000000000)
model.setParam("TimeLimit", SOFT_TIME_LIMIT)
model.update()
print(f"Sākuma risinājums saglabāts. Statuss: {model.status}, risinājumi: {model.SolCount}")


## Mīkstie ierobežojumi

### S0. Svari


In [ ]:
w_unfavorable = CURRENT_EXPERIMENT["unfavorable"]
w_teacher_gap = CURRENT_EXPERIMENT["teacher_gap"]
w_class_gap = CURRENT_EXPERIMENT["class_gap"]
w_subject_spread = CURRENT_EXPERIMENT["subject_spread"]
w_teacher_min_day = CURRENT_EXPERIMENT["teacher_min_day"]



### S1. Skolotāja nevēlamie laiki


In [ ]:
teacher_busy = model.addVars(T, D, P, vtype=GRB.BINARY, name="teacher_busy")

for t, lesson_ids in teacher_lesson_ids.items():
    lesson_ids = sorted(set(lesson_ids))
    for d in D:
        for p in P:
            teacher_time_vars = []
            for l_id in lesson_ids:
                teacher_time_vars.extend(x.select(l_id, '*', d, p))
            if teacher_time_vars:
                model.addConstr(
                    teacher_busy[t, d, p] == gp.quicksum(teacher_time_vars),
                    name=f"S1_teacher_busy_link_{t}_{d}_{p}"
                )
            else:
                model.addConstr(
                    teacher_busy[t, d, p] == 0,
                    name=f"S1_teacher_busy_zero_{t}_{d}_{p}"
                )

unfavorable_penalty_terms = [
    teacher_busy[t, d, p]
    for t in T for d in D for p in P
    if Unf.get((t, d, p), 0) == 1
]

model.update()


### S2. Skolotāja logi dienas vidū


In [ ]:
teacher_gap = model.addVars(T, D, P, vtype=GRB.BINARY, name="teacher_gap")

for t in T:
    for d in D:
        for p in P:
            if p == min(P) or p == max(P):
                model.addConstr(teacher_gap[t, d, p] == 0, name=f"S2_teacher_gap_edge_{t}_{d}_{p}")
            else:
                model.addConstr(
                    teacher_gap[t, d, p] >= teacher_busy[t, d, p - 1] + teacher_busy[t, d, p + 1] - 1 - teacher_busy[t, d, p],
                    name=f"S2_teacher_gap_{t}_{d}_{p}"
                )


### S3. Klases logi dienas vidū

In [ ]:
class_gap = model.addVars(C, D, P, vtype=GRB.BINARY, name="class_gap")

for c in C:
    for d in D:
        for p in P:
            if p == min(P) or p == max(P):
                model.addConstr(class_gap[c, d, p] == 0, name=f"S3_class_gap_edge_{c}_{d}_{p}")
            else:
                model.addConstr(
                    class_gap[c, d, p] >= class_busy[c, d, p - 1] + class_busy[c, d, p + 1] - 1 - class_busy[c, d, p],
                    name=f"S3_class_gap_{c}_{d}_{p}"
                )


### S4. Priekšmeta sadalījums pa dienām


In [ ]:
subject_day_over = []
for (c, s), req in Req.items():
    lesson_ids = [
        l_id for l_id in class_lesson_ids_by_class.get(c, [])
        if lesson_subjects[l_id] == s
    ]
    if not lesson_ids or req <= 1:
        continue
    target_per_day = max(1, (req + len(D) - 1) // len(D))
    for d in D:
        subject_day_vars = []
        for l_id in lesson_ids:
            for p in P:
                subject_day_vars.extend(x.select(l_id, '*', d, p))
        if subject_day_vars:
            over = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name=f"subject_day_over_{c}_{s}_{d}")
            model.addConstr(
                over >= gp.quicksum(subject_day_vars) - target_per_day,
                name=f"S4_subject_spread_{c}_{s}_{d}"
            )
            subject_day_over.append(over)


### S5. Skolotāja minimālā dienas slodze

In [ ]:
teacher_day_active = model.addVars(T, D, vtype=GRB.BINARY, name="teacher_day_active")
teacher_min_day_under = model.addVars(T, D, lb=0, vtype=GRB.CONTINUOUS, name="teacher_min_day_under")

for t in T:
    for d in D:
        day_load = gp.quicksum(teacher_busy[t, d, p] for p in P)
        model.addConstr(
            teacher_day_active[t, d] >= day_load / MaxDay[t],
            name=f"S5_teacher_day_active_{t}_{d}"
        )
        model.addConstr(
            teacher_min_day_under[t, d] >= MinDay[t] * teacher_day_active[t, d] - day_load,
            name=f"S5_teacher_min_day_under_{t}_{d}"
        )


### Mērķa funkcija


In [ ]:
obj = (
    w_unfavorable * gp.quicksum(unfavorable_penalty_terms)
    + w_teacher_gap * gp.quicksum(teacher_gap[t, d, p] for t in T for d in D for p in P)
    + w_class_gap * gp.quicksum(class_gap[c, d, p] for c in C for d in D for p in P)
    + w_subject_spread * gp.quicksum(subject_day_over)
    + w_teacher_min_day * gp.quicksum(teacher_min_day_under[t, d] for t in T for d in D)
)

model.setObjective(obj, GRB.MINIMIZE)
model.update()


In [ ]:
model.optimize()

In [ ]:
experiment_name = globals().get("EXPERIMENT_NAME", "run")

EXPERIMENT_FIELDS = [
    "experiment",
    "model_status",
    "solutions",
    "objective_value",
    "mip_gap",
    "runtime_seconds",
    "solver_runtime_seconds",
    "scheduled_lessons",
    "unique_lessons",
    "class_parallel_same_subject_slots",
    "unfavorable_lessons",
    "teacher_gaps",
    "class_gaps",
    "teacher_max_day_over",
    "teacher_min_day_under",
    "subject_spread_over",
    "soft_score",
]


def split_participants(value):
    return [part for part in str(value or "").split("+") if part]


def count_gaps(periods):
    if not periods:
        return 0
    periods = sorted(periods)
    return sum(1 for p in range(periods[0], periods[-1] + 1) if p not in periods)


def load_unfavorable_slots(path):
    if not os.path.exists(path):
        return set()
    rows = pd.read_csv(path)
    return {
        (row.teacher_id, int(row.day), int(row.period))
        for row in rows.itertuples(index=False)
        if int(row.unfavorable) == 1
    }


def load_teacher_limits(path):
    if not os.path.exists(path):
        return {}
    rows = pd.read_csv(path)
    return {
        row.teacher_id: {"max_day": int(row.max_day), "min_day": int(row.min_day)}
        for row in rows.itertuples(index=False)
    }


def evaluate_schedule(schedule_df):
    unfavorable_slots = load_unfavorable_slots("../data/teacher_unfavorable.csv")
    teacher_limits = load_teacher_limits("../data/teacher_load.csv")

    teacher_slots = defaultdict(list)
    class_slots = defaultdict(list)
    class_subject_day = defaultdict(int)

    for row in schedule_df.itertuples(index=False):
        day = int(row.day)
        period = int(row.period)
        teacher_slots[(row.teacher, day, period)].append(row.lesson_id)

        for class_id in split_participants(row.participants):
            class_slots[(class_id, day, period)].append((row.subject, row.lesson_id))
            class_subject_day[(class_id, row.subject, day)] += 1

    class_parallel_same_subject_slots = sum(
        1
        for values in class_slots.values()
        if len(values) > 1 and len({subject for subject, _ in values}) == 1
    )

    unfavorable_lessons = sum(
        1
        for row in schedule_df.itertuples(index=False)
        if (row.teacher, int(row.day), int(row.period)) in unfavorable_slots
    )

    teacher_day_periods = defaultdict(set)
    class_day_periods = defaultdict(set)
    for teacher, day, period in teacher_slots:
        teacher_day_periods[(teacher, day)].add(period)
    for class_id, day, period in class_slots:
        class_day_periods[(class_id, day)].add(period)

    teacher_gaps = sum(count_gaps(periods) for periods in teacher_day_periods.values())
    class_gaps = sum(count_gaps(periods) for periods in class_day_periods.values())

    teacher_max_day_over = 0
    teacher_min_day_under = 0
    for (teacher, day), periods in teacher_day_periods.items():
        limits = teacher_limits.get(teacher)
        if limits is None:
            continue
        teacher_max_day_over += max(0, len(periods) - limits["max_day"])
        teacher_min_day_under += max(0, limits["min_day"] - len(periods))

    by_class_subject = defaultdict(list)
    for (class_id, subject, day), count in class_subject_day.items():
        by_class_subject[(class_id, subject)].append(count)

    subject_spread_over = 0
    for counts in by_class_subject.values():
        total = sum(counts)
        active_days = len(counts)
        if total <= 1:
            continue
        target = max(1, (total + active_days - 1) // active_days)
        subject_spread_over += sum(max(0, count - target) for count in counts)

    metrics = {
        "scheduled_lessons": len(schedule_df),
        "unique_lessons": schedule_df["lesson_id"].nunique(),
        "class_parallel_same_subject_slots": class_parallel_same_subject_slots,
        "unfavorable_lessons": unfavorable_lessons,
        "teacher_gaps": teacher_gaps,
        "class_gaps": class_gaps,
        "teacher_max_day_over": teacher_max_day_over,
        "teacher_min_day_under": teacher_min_day_under,
        "subject_spread_over": subject_spread_over,
    }
    metrics["soft_score"] = (
        metrics["unfavorable_lessons"]
        + metrics["teacher_gaps"]
        + metrics["class_gaps"]
        + metrics["teacher_min_day_under"]
        + metrics["subject_spread_over"]
    )
    return metrics


def save_experiment_result(path, experiment_name, metrics):
    if os.path.exists(path):
        df = pd.read_csv(path)
    else:
        df = pd.DataFrame(columns=EXPERIMENT_FIELDS)

    row = {field: "" for field in EXPERIMENT_FIELDS}
    row.update({"experiment": experiment_name, **metrics})

    if "experiment" in df.columns:
        df = df[df["experiment"] != experiment_name]
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df = df.reindex(columns=EXPERIMENT_FIELDS)
    df.to_csv(path, index=False, encoding="utf-8")


print(f"Statuss: {model.status}")
print(f"Risinājumi: {model.SolCount}")

if model.SolCount > 0:
    chosen_variants = {
        key: variant
        for key, variant in variant_choices
        if y[key, variant].X > 0.5
    }

    expected_lessons = sum(
        1
        for l_id in unique_lesson_ids
        if not VariantId.get(l_id, "")
        or chosen_variants.get(PartitionKey[l_id]) == VariantId[l_id]
    )

    schedule_rows = []
    for l in L:
        l_id = l[0]
        for r in R:
            for d in D:
                for p in P:
                    var = x.get((l_id, r, d, p))
                    if var is not None and var.X > 0.5:
                        schedule_rows.append({
                            "lesson_id": l_id,
                            "class": l[1],
                            "participants": "+".join(Participants.get(l_id, (l[1],))),
                            "subject": l[2],
                            "teacher": l[4],
                            "room": r,
                            "day": d,
                            "period": p,
                            "shared_key": l[3],
                            "share_mode": ShareMode.get(l_id, ""),
                            "group_index": GroupIndex.get(l_id, ""),
                            "instance": Instance.get(l_id, ""),
                            "stream_id": StreamId.get(l_id, ""),
                            "partition_key": PartitionKey.get(l_id, ""),
                            "variant_id": VariantId.get(l_id, ""),
                        })

    os.makedirs("../results/experiments", exist_ok=True)

    schedule_df = pd.DataFrame(schedule_rows).sort_values(["day", "period", "class", "subject", "teacher"])
    variants_df = pd.DataFrame(sorted(chosen_variants.items()), columns=["partition_key", "variant_id"])

    schedule_df.to_csv("../results/timetable.csv", index=False, encoding="utf-8")
    variants_df.to_csv("../results/selected_partition_variants.csv", index=False, encoding="utf-8")
    schedule_df.to_csv(f"../results/experiments/timetable_{experiment_name}.csv", index=False, encoding="utf-8")
    variants_df.to_csv(f"../results/experiments/selected_partition_variants_{experiment_name}.csv", index=False, encoding="utf-8")

    experiment_metrics = {
        "model_status": model.status,
        "solutions": model.SolCount,
        "objective_value": model.ObjVal,
        "mip_gap": model.MIPGap,
        "runtime_seconds": time.perf_counter() - experiment_start_time,
        "solver_runtime_seconds": model.Runtime,
        **evaluate_schedule(schedule_df),
    }
    save_experiment_result("../results/experiment_results.csv", experiment_name, experiment_metrics)

    print(f"Nodarbības: {len(schedule_rows)}/{expected_lessons}")
    print(f"soft_score: {experiment_metrics['soft_score']}")
    print(f"Mērķis: {model.ObjVal:.2f}")
    print(f"MIP gap: {model.MIPGap:.4f}")
    print("Saglabāts: results/timetable.csv")
    print("Saglabāts: results/experiment_results.csv")

elif model.status == GRB.INFEASIBLE:
    model.computeIIS()
    for constr in model.getConstrs():
        if constr.IISConstr:
            print(constr.ConstrName)
else:
    print("Risinājums nav atrasts")

